# Reproducing Kobayashi et al. (2024)
*Time series generation for option pricing on quantum computers using tensor networks.*

**arXiv:2402.17148**

Pipeline:
1. Heston model simulation
2. Discretisation with `BasisEncoder`
3. MPS training
4. Sequential sampling
5. European / Asian / Lookback / Barrier option pricing
6. Comparison with Black-Scholes implied volatility

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))
Pkg.resolve()
Pkg.instantiate()

using MPSFast
using MPSFast.Encoders
using Plots
using Random, LinearAlgebra, Statistics, Printf, Distributions

## 1. Heston Model Simulation

In [ ]:
"""
    simulate_heston(N, M; S0, V0, κ, θ, ξ, ρ, r, dt, rng)

Generate `N` paths of `M` steps under the Heston stochastic volatility model
using a simple Euler-Maruyama discretisation.
"""
function simulate_heston(
    N::Int, M::Int;
    S0 = 100.0, V0 = 0.04, κ = 2.0, θ = 0.04,
    ξ = 0.3, ρ = -0.7, r = 0.0, dt = 1/252,
    rng = Random.default_rng(),
)
    S = Matrix{Float64}(undef, N, M)
    V = Matrix{Float64}(undef, N, M)
    for i in 1:N
        s = S0;  v = V0
        for t in 1:M
            z1 = randn(rng);  z2 = randn(rng)
            w1 = z1
            w2 = ρ * z1 + sqrt(1 - ρ^2) * z2
            v  = max(v + κ * (θ - v) * dt + ξ * sqrt(max(v, 0.0) * dt) * w2, 0.0)
            s  = s * exp((r - 0.5 * v) * dt + sqrt(max(v, 0.0) * dt) * w1)
            S[i, t] = s
            V[i, t] = v
        end
    end
    return S, V
end

rng = MersenneTwister(2024)
N_train, M = 10_000, 10

paths, paths_ν = simulate_heston(N_train, M; rng = rng)
println("Heston paths: ", size(paths))
println("Mean S[M]  : ", round(mean(paths[:, M]), digits=2))
println("Std  S[M]  : ", round(std(paths[:, M]),  digits=2))

In [ ]:
# ── Figure: Heston sample trajectories (S and σ = √ν) ────────────────────────
σ_paths = sqrt.(max.(paths_ν, 0.0))

p_hp = plot(layout=(2,1), size=(720, 560), plot_title="Heston Trajectories", dpi=150)
for i in 1:200
    plot!(p_hp[1], 1:M, paths[i,:];    color=:steelblue, alpha=0.25, label=false)
    plot!(p_hp[2], 1:M, σ_paths[i,:];  color=:darkgreen, alpha=0.25, label=false)
end
plot!(p_hp[1], 1:M, vec(mean(paths;    dims=1)); color=:black, lw=2.5, label="mean S",
      xlabel="step t", ylabel="S", title="Price paths Sₜ", legend=:bottomleft)
plot!(p_hp[2], 1:M, vec(mean(σ_paths;  dims=1)); color=:black, lw=2.5, label="mean σ",
      xlabel="step t", ylabel="σ = √ν", title="Volatility σₜ = √νₜ", legend=:topleft)
display(p_hp)

In [ ]:
# ── Figure: Heston mean ± std over time ──────────────────────────────────────
μS   = vec(mean(paths;    dims=1))
σS_s = vec(std(paths;     dims=1))
μσ   = vec(mean(σ_paths;  dims=1))
σσ   = vec(std(σ_paths;   dims=1))

p_ms = plot(layout=(2,1), size=(720, 520), plot_title="Heston statistics")
plot!(p_ms[1], 1:M, μS;  ribbon=σS_s, fillalpha=0.25, color=:steelblue, lw=2,
      label="mean ± std", title="Sₜ", xlabel="step t", ylabel="S")
plot!(p_ms[2], 1:M, μσ;  ribbon=σσ,   fillalpha=0.25, color=:darkgreen, lw=2,
      label="mean ± std", title="σₜ = √νₜ", xlabel="step t", ylabel="σ")
display(p_ms)

## 2. Encode with BasisEncoder

In [ ]:
m   = 5    # d = 2^4 = 16 buckets per timestep
enc = BasisEncoder(m)
fit_grid!(enc, paths)
xi  = encode_paths(enc, paths)

encoder_summary(enc, M, 200)
println("xi range: ", extrema(xi))

In [ ]:
# ── Figure: Continuous paths vs. discretised buckets ─────────────────────────
n_show  = min(200, N_train)
d_phys  = site_dim(enc)

p_disc = plot(layout=(1,2), size=(800, 360),
              plot_title="Discretization (m=$(m) bits, d=$(d_phys) levels)")
for i in 1:n_show
    plot!(p_disc[1], 1:M, paths[i,:];
          color=:steelblue, alpha=0.10, legend=false)
    plot!(p_disc[2], 1:M, xi[i,:];
          linetype=:steppost, color=:orange, alpha=0.5, legend=false)
end
for lv in 1:max(1, d_phys ÷ 8):d_phys
    hline!(p_disc[2], [lv]; color=:gray, alpha=0.4, ls=:dot, legend=false)
end
xlabel!(p_disc[1], "step t"); ylabel!(p_disc[1], "S")
xlabel!(p_disc[2], "step t"); ylabel!(p_disc[2], "bucket index")
title!(p_disc[1], "Continuous"); title!(p_disc[2], "Discretised")
display(p_disc)

## 3. Train MPS

In [ ]:
D_max    = 150
n_epochs = 100
η        = 5e-4
ε_cut    = 1e-5

# ── Validation split (10 % held out for early-stopping monitor) ────────────
val_frac = 0.10
N_total  = size(xi, 1)
N_val    = round(Int, val_frac * N_total)
perm     = randperm(MersenneTwister(0), N_total)
xi_val   = xi[perm[1:N_val],        :]
xi_tr    = xi[perm[N_val+1:end],    :]
println("Train: $(size(xi_tr,1)) paths   Val: $(size(xi_val,1)) paths")

mps = init_mps(chain_length(enc, M), site_dim(enc), D_max; rng = MersenneTwister(1))

bond_log    = []
val_nll_log = Float64[]
nll_hist = train_mps!(
    mps, xi_tr, n_epochs, η, D_max, ε_cut;
    verbose          = true,
    nll_samples      = 10_000,
    bond_log         = bond_log,
    checkpoint_dir   = "checkpoints",
    checkpoint_every = 5,
    lr_schedule      = cosine_lr,
    val_data         = xi_val,
    val_samples      = 2_000,
    patience         = 15,
    val_nll_log      = val_nll_log,
)

In [ ]:
# NLL convergence
println("NLL history:")
for (e, nll) in enumerate(nll_hist)
    @printf "  epoch %2d  NLL = %.4f\n" e nll
end

# Entanglement
_, entr = bipartite_entropies(mps)
println("\nBipartite entropies (nats): ", round.(entr, digits=3))

In [ ]:
# ── Figure: Training convergence — NLL + bipartite entropy per bond ──────────
H = entropy_history(bond_log, chain_length(enc, M), n_epochs)
M_bonds = size(H, 1)
n_done  = length(nll_hist)   # may be < n_epochs if early stopping fired

p_conv = plot(layout=(2,1), size=(720, 520), plot_title="Training convergence")

plot!(p_conv[1], 1:n_done, nll_hist;
      xlabel="Epoch", ylabel="NLL (nats)",
      marker=:circle, ms=3, color=:steelblue, lw=2,
      label="train")
if !isempty(val_nll_log)
    plot!(p_conv[1], 1:length(val_nll_log), val_nll_log;
          marker=:circle, ms=3, color=:tomato, lw=2, ls=:dash,
          label="val")
end
plot!(p_conv[1]; legend=:topright, title="NLL vs. epoch")

colors = cgrad(:viridis, M_bonds; categorical=true)
for b in 1:M_bonds
    plot!(p_conv[2], 1:n_done, H[b, 1:n_done];
          xlabel="Epoch", ylabel="S(b) (nats)",
          marker=:circle, ms=2, lw=1.5,
          color=colors[b], label="b=$b")
end
hline!(p_conv[2], [log(D_max)]; color=:red, ls=:dash, lw=1.5,
       label="log Dmax")
plot!(p_conv[2]; legend=:outerright, title="Bipartite entropy per cut b")
display(p_conv)

@printf "\nEntropy summary (last epoch):\n"
@printf "%-6s %12s %12s %15s\n" "bond" "S(b) [nats]" "exp(S)" "S / log Dmax"
for b in 1:M_bonds
    sb = H[b, n_done]
    @printf "%-6d %12.4f %12.2f %15.3f\n" b sb exp(sb) sb / log(Float64(D_max))
end

## 4. Sample New Paths

In [ ]:
N_samp = 5_000
sampled_paths, sampled_xi = sample_paths(enc, mps, N_samp; seed = 7)

println("Sampled paths shape: ", size(sampled_paths))

# ── Marginal table ─────────────────────────────────────────────────────────────
println("\n  t   Train mean  Samp mean   Train std   Samp std")
for t in filter(t -> t <= M, [1, 5, 10, 20, 30])
    @printf("  %-3d  %9.3f  %9.3f  %9.3f  %9.3f\n", t,
            mean(paths[:,t]), mean(sampled_paths[:,t]),
            std(paths[:,t]),  std(sampled_paths[:,t]))
end

# ── Figure 1: example paths ────────────────────────────────────────────────────
ts = (1:M) ./ 252
p_paths = plot(ts, sampled_paths[1:80, :]';
    color = :steelblue, alpha = 0.25, lw = 0.8, label = false,
    xlabel = "time (years)", ylabel = "price",
    title  = "80 MPS-sampled Heston paths", size = (700, 320))
display(p_paths)

# ── Figure 2: per-timestep marginals at selected times ──────────────────────────
d_enc = site_dim(enc)
times_show = filter(t -> t <= M, [1, 5, 10, 20, 30])
train_marg = [count(==(σ), xi[:,        t]) / N_train for σ in 1:d_enc, t in 1:M]
samp_marg  = [count(==(σ), sampled_xi[:, t]) / N_samp  for σ in 1:d_enc, t in 1:M]

p_marg = plot(layout = (1, length(times_show)),
              size   = (200 * length(times_show), 280),
              title  = reshape(["t = $t" for t in times_show], 1, :),
              legend = false)
for (k, t) in enumerate(times_show)
    bar!(p_marg, subplot=k, 1:d_enc, train_marg[:, t];
         fillalpha=0.55, color=:steelblue, label="train",
         xlabel="bucket σ", ylabel = k==1 ? "P(σ)" : "")
    bar!(p_marg, subplot=k, 1:d_enc, samp_marg[:,  t];
         fillalpha=0.55, color=:tomato,    label="MPS")
end
display(p_marg)

# ── Figure 3: marginal mean ± std over time ─────────────────────────────────────
train_mean_t = vec(mean(paths,        dims=1))
samp_mean_t  = vec(mean(sampled_paths, dims=1))
train_std_t  = vec(std(paths,         dims=1))
samp_std_t   = vec(std(sampled_paths,  dims=1))

p_stats = plot(ts, train_mean_t; ribbon=train_std_t, fillalpha=0.2,
    label="Heston (train)", color=:steelblue, lw=2,
    xlabel="time (years)", ylabel="price",
    title="Marginal mean ± std", size=(700, 320), legend=:topleft)
plot!(p_stats, ts, samp_mean_t; ribbon=samp_std_t, fillalpha=0.2,
    label="MPS sample", color=:tomato, lw=2, ls=:dash)
display(p_stats)

# ── Figure 4: NLL learning curve ────────────────────────────────────────────────
p_nll = plot(1:length(nll_hist), nll_hist;
    marker=:circle, markersize=3, color=:darkorange, lw=2,
    xlabel="epoch", ylabel="NLL (nats / sample)",
    title="Training NLL — Heston MPS", legend=false, size=(600, 300))
display(p_nll)

In [ ]:
# ── Figure: Bipartite von Neumann entropy of trained MPS ─────────────────────
# One right-to-left SVD sweep on the MPS to get S(b) for every bipartition b.
Svals_mps, Svn = bipartite_entropies(mps)
br = 1:length(Svn)

d_phys = site_dim(enc)
logd   = log(Float64(d_phys))
# Normalised S(b) / (b · log d) ∈ [0,1] — measures entanglement relative to
# the maximum entropy of b sites each with dimension d.
Svn_norm = [isfinite(Svn[b]) ? Svn[b] / (b * logd) : NaN for b in br]

# Recompute Heston means for the plot (may already exist from earlier cells)
σ_paths_l = sqrt.(max.(paths_ν, 0.0))
μS_l  = vec(mean(paths;      dims=1))
μσ_l  = vec(mean(σ_paths_l;  dims=1))

p_ent = plot(layout=(3,1), size=(720, 820),
             plot_title="Heston means & MPS bipartite entropy")

plot!(p_ent[1], 1:M, μS_l;  color=:steelblue, lw=2, marker=:circle, ms=3,
      legend=false, title="⟨Sₜ⟩", xlabel="step t", ylabel="⟨S⟩")

plot!(p_ent[2], 1:M, μσ_l;  color=:darkgreen, lw=2, marker=:circle, ms=3,
      legend=false, title="⟨σₜ⟩ = ⟨√νₜ⟩", xlabel="step t", ylabel="⟨σ⟩")

plot!(p_ent[3], collect(br), Svn_norm;
      color=:purple, lw=2, marker=:diamond, ms=4,
      label="S / (b log d)", legend=:bottomright,
      title="Normalised bipartite entropy",
      xlabel="bipartition 1…b | rest", ylabel="S / (b log d)", ylims=(0, 1.05))

p3r = twinx(p_ent[3])
plot!(p3r, collect(br), Svn[br];
      color=:gray, lw=1.5, marker=:circle, ms=3, alpha=0.85,
      label="S (nats)", legend=:topleft, ylabel="S (nats)")

display(p_ent)

@printf "\nFinal bipartite entropy (nats):\n"
@printf "%-6s %14s %14s %14s\n" "bond" "S(b) [nats]" "exp(S(b))" "S/(b log d)"
for b in br
    @printf "%-6d %14.4f %14.2f %14.4f\n" b Svn[b] exp(Svn[b]) Svn_norm[b]
end

## 5. Option Pricing

In [ ]:
S0 = 100.0
r  = 0.0
T  = M / 252

"""Black-Scholes call price."""
function bs_call(S, K, T, r, σ)
    T <= 0 && return max(S - K, 0.0)
    d1 = (log(S/K) + (r + 0.5*σ^2)*T) / (σ*sqrt(T))
    d2 = d1 - σ*sqrt(T)
    S * cdf(Normal(), d1) - K * exp(-r*T) * cdf(Normal(), d2)
end

"""Newton's method to find implied vol."""
function implied_vol(price, S, K, T, r; tol=1e-8, max_iter=100)
    σ = 0.2
    for _ in 1:max_iter
        p  = bs_call(S, K, T, r, σ)
        d1 = (log(S/K) + (r + 0.5*σ^2)*T) / (σ*sqrt(T))
        vg = S * sqrt(T/(2π)) * exp(-0.5*d1^2)   # vega
        vg < 1e-12 && break
        σ_new = σ - (p - price) / vg
        abs(σ_new - σ) < tol && return σ_new
        σ = clamp(σ_new, 0.001, 5.0)
    end
    return σ
end

# Price options at several strikes
strikes = S0 .* [0.80, 0.90, 0.95, 1.00, 1.05, 1.10, 1.20]

println("European Call Options (MPS Monte Carlo vs Black-Scholes):")
println("  Strike   MC Price   BS Price (σ=0.20)   Impl. Vol")
for K in strikes
    # Discounted payoff under MPS
    payoffs = max.(sampled_paths[:, end] .- K, 0.0)
    mc_price = exp(-r*T) * mean(payoffs)
    bs_price = bs_call(S0, K, T, r, 0.20)
    ivol = implied_vol(mc_price, S0, K, T, r)
    @printf "  %6.1f   %8.4f   %8.4f             %6.4f\n" K mc_price bs_price ivol
end

In [ ]:
# ── Figure 5: Implied-vol smile ───────────────────────────────────────────────
smile_strikes = S0 .* range(0.75, 1.30; length = 30)
mc_prices   = [exp(-r*T) * mean(max.(sampled_paths[:, end] .- K, 0.0)) for K in smile_strikes]
ivols       = [implied_vol(p, S0, K, T, r) for (p, K) in zip(mc_prices, smile_strikes)]
bs_flat_iv  = fill(0.20, length(smile_strikes))

p_smile = plot(smile_strikes, ivols .* 100;
    lw=2, marker=:circle, markersize=3, color=:darkorange,
    label="MPS implied vol", xlabel="strike K",
    ylabel="implied vol (%)", title="Implied-vol smile — MPS vs flat 20%",
    size=(650, 320), legend=:top)
hline!(p_smile, [20.0]; lw=1.5, ls=:dash, color=:gray, label="20% flat")
display(p_smile)

In [ ]:
# Asian options (arithmetic mean payoff)
println("\nAsian Call Options (K = S0):")
K = S0
asian_payoffs = max.(mean(sampled_paths, dims=2)[:] .- K, 0.0)
@printf "  Asian Call price = %.4f\n" mean(asian_payoffs)

# Lookback options (maximum payoff)
lb_payoffs = max.(maximum(sampled_paths, dims=2)[:] .- S0, 0.0)
@printf "  Lookback Call price = %.4f\n" mean(lb_payoffs)

# Barrier option (up-and-out)
B_level = 1.2 * S0
survived = all(sampled_paths .< B_level, dims=2)[:]
barrier_payoffs = max.(sampled_paths[:, end] .- S0, 0.0) .* survived
@printf "  Up-and-Out Call (B=%.0f) price = %.4f\n" B_level mean(barrier_payoffs)

## 6. Checkpoint I/O

In [ ]:
# Save the final model
meta = Dict{String,Any}(
    "N_train" => N_train, "M" => M, "m" => m,
    "D_max" => D_max, "n_epochs" => n_epochs,
    # Encoder grid — needed to reconstruct enc in downstream notebooks
    "encoder" => string(typeof(enc)),
    "Smin"    => enc.Smin,
    "Smax"    => enc.Smax,
)
save_mps_bundle("mps_heston.jld2", mps, nll_hist, n_epochs, meta; bond_log = bond_log)
println("Saved to mps_heston.jld2")

# Reload check
mps2, nll2, epoch2, meta2 = load_mps_bundle("mps_heston.jld2")
println("Reloaded: epoch=", epoch2, "  NLL=", round(nll2[end], digits=4))